# Reference resolution → OpenIE

Run from the installed project environment. Model calls are opt-in and require provider credentials in the process environment. No credentials are saved here.

In [ ]:
from data_platform.digest.openie import extract_claims

markdown_doc = "# Report\n\n## Factory\n\nAcme builds cars. It plans buses in 2027."
RUN_LIVE = False

In [ ]:
if RUN_LIVE:
    result = await extract_claims(markdown_doc)
    for sentence in result.sentences:
        print(sentence.text, "→", sentence.resolution.text, sentence.status)
    for claim in result.claims:
        print(claim.subject, claim.relation, claim.object, claim.qualifiers)
        print(claim.text, (claim.start, claim.end))
    display(result.model_dump(mode="json"))

## Inject a dedicated backend

Implement async `resolve(*, title, section)` returning `ResolutionResponse`, or async `extract(*, sentence, resolved)` returning `ExtractionResponse`. Load your model outside these methods. For blocking inference use `asyncio.to_thread`; serialize access if the model is not thread-safe. Adapters own their model configuration and must include `BackendMetadata`.

In [ ]:
from data_platform.digest.openie import BackendMetadata, ExtractionResponse

class DedicatedExtractor:
    def __init__(self, infer):
        self.infer = infer  # async callable producing Proposition records

    async def extract(self, *, sentence, resolved):
        propositions = await self.infer(resolved.text)
        return ExtractionResponse(
            propositions=propositions,
            metadata=BackendMetadata(backend="dedicated-model"),
        )

# result = await extract_claims(markdown_doc, extractor=DedicatedExtractor(infer))

Inspect section-level resolution traces in `result.resolution_stages` and extraction traces in `result.sentences[*].stages`. Offsets always refer to original source sentences. Review semantic accuracy separately; typed outputs do not prove entailment.